# UMA gas-phase TS3 review

The final attempt starts from the **ALPB-optimized TS3 geometry**, displaces
its two shallow peripheral imaginary modes, then runs **gas-phase UMA OptTS
and NumFreq**. The seed is not a gas-phase profile reference; only the new
gas-phase optimized coordinates can become one.

Run this notebook with the **UMA conda environment** after the submitted job
has finished. The first code cell copies its result from the cluster. The
notebook shows termination, all negative frequencies, reactive-core distances,
and the animated mode. Accept TS3 only if OptTS and NumFreq terminated, there
is **exactly one** negative frequency, and its motion follows the intended
TS3 reactive coordinate. Otherwise keep TS3 quarantined.

In [1]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display
import frust as ft

REMOTE = (
    "HPC5:/lustre/hpc/kemi/jmni/results/"
    "uma-task04-ts3-gas-final-20260930/workflow/ts3_gas/final.parquet"
)
LOCAL = Path(
    "/Users/jacobmolinnielsen/Developer/FrustActivationProject/fruits/"
    "results-2026/uma-task04-ts3-gas-final-20260930/final.parquet"
)
LOCAL.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["scp", "-q", REMOTE, str(LOCAL)], check=True)
result = pd.read_parquet(LOCAL)
assert len(result) == 1, f"Expected one TS3 row, got {len(result)}"
row = result.iloc[0]
print("Copied:", LOCAL)
print("State:", row["state_id"], "Source:", row["source_xyz"])
print("OptTS normal termination:", bool(row["uma_ts_opt-NT"]))
print("NumFreq normal termination:", bool(row["uma_freq-NT"]))
for stage in ("uma_ts_opt", "uma_freq"):
    error = row.get(f"{stage}-error")
    if isinstance(error, str) and error:
        print(f"{stage} error: {error}")

Copied: /Users/jacobmolinnielsen/Developer/FrustActivationProject/fruits/results-2026/uma-task04-ts3-gas-final-20260930/final.parquet
State: TS3 Source: ts3_TMP.xyz
OptTS normal termination: True
NumFreq normal termination: True


In [2]:
raw_modes = row.get("uma_freq-vibs")
modes = list(raw_modes) if isinstance(raw_modes, (list, tuple, np.ndarray)) else []
frequency_table = pd.DataFrame(
    {"mode_index": range(len(modes)),
     "frequency_cm1": [float(mode["frequency"]) for mode in modes]}
)
negative = frequency_table.loc[frequency_table.frequency_cm1 < 0]
display(frequency_table.head(12))
print("All imaginary modes:")
display(negative)
print("Exactly one imaginary mode:", len(negative) == 1)

previous = Path(
    "/Users/jacobmolinnielsen/Developer/FrustActivationProject/fruits/"
    "results-2026/uma-task04-references-20260929/ts3_gas.parquet"
)
if previous.exists():
    old = pd.read_parquet(previous).iloc[0]
    print("Original gas TS3 negatives:",
          [float(m["frequency"]) for m in old["uma_freq-vibs"]
           if float(m["frequency"]) < 0])

,mode_index,frequency_cm1
0,0,-94.46
1,1,26.02
2,2,34.03
3,3,38.66
4,4,47.83
5,5,64.47
6,6,74.53
7,7,86.05
8,8,88.23
9,9,100.85


All imaginary modes:


,mode_index,frequency_cm1
0,0,-94.46


Exactly one imaginary mode: True
Original gas TS3 negatives: [-27.81, -13.02]


In [3]:
if bool(row["uma_ts_opt-NT"]):
    coords = np.vstack(row["uma_ts_opt-oc"]).astype(float)
    roles = {name: int(index) for name, index in row["constraint_roles"].items()}
    pairs = [
        ("transfer_H", "cat_B"),
        ("transfer_H", "pin_B"),
        ("transfer_H", "substrate_C"),
        ("cat_B", "substrate_C"),
        ("pin_B", "substrate_C"),
        ("pin_B", "cat_B"),
    ]
    def distance(xyz, first, second):
        return float(np.linalg.norm(xyz[roles[first]] - xyz[roles[second]]))

    if len(negative) == 1:
        mode_index = int(negative.iloc[0]["mode_index"])
        displacement = np.vstack(modes[mode_index]["mode"]).astype(float)
        before, after = coords - 0.25 * displacement, coords + 0.25 * displacement
    else:
        mode_index = None
        before = after = coords
    display(pd.DataFrame([
        {"pair": f"{first}–{second}",
         "optimized_A": distance(coords, first, second),
         "mode_minus_A": distance(before, first, second),
         "mode_plus_A": distance(after, first, second)}
        for first, second in pairs
    ]).round(3))
else:
    print("No optimized geometry: inspect the OptTS error above.")

,pair,optimized_A,mode_minus_A,mode_plus_A
0,transfer_H–cat_B,1.229,1.217,1.242
1,transfer_H–pin_B,1.590,1.700,1.481
2,transfer_H–substrate_C,2.408,2.444,2.374
3,cat_B–substrate_C,1.879,1.955,1.804
4,pin_B–substrate_C,1.624,1.595,1.655
5,pin_B–cat_B,2.049,2.139,1.962


In [4]:
if bool(row["uma_ts_opt-NT"]) and len(negative) == 1:
    viewer_path = LOCAL.parent / "ts3_gas_reactive_mode.html"
    viewer = ft.plot_vibs(
        result,
        row_index=0,
        vId=mode_index,
        custom_coords_col_name="uma_ts_opt-oc",
        export_HTML=str(viewer_path),
        width=700,
        height=500,
        reps=100,
    )
    print("Saved animated viewer:", viewer_path)
    display(viewer)
else:
    print("No unique reactive imaginary mode to animate.")

vibs col uma_freq-vibs
coords col uma_ts_opt-oc
HTML export successful: /Users/jacobmolinnielsen/Developer/FrustActivationProject/fruits/results-2026/uma-task04-ts3-gas-final-20260930/ts3_gas_reactive_mode.html
Saved animated viewer: /Users/jacobmolinnielsen/Developer/FrustActivationProject/fruits/results-2026/uma-task04-ts3-gas-final-20260930/ts3_gas_reactive_mode.html


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Independent TS3 versus TS4 check (2026-09-30)

The run satisfies the numerical first-order-saddle check: both calculations
terminated, and mode 0 is the only imaginary frequency (**−94.46 cm⁻¹**).
However, the optimized reactive-core distances resemble the separately
optimized **gas TS4** reference much more than the TS3 seed:

| Geometry | Bcat–H (Å) | Bpin–H (Å) | Bcat–Csub (Å) | Bpin–Csub (Å) |
| --- | ---: | ---: | ---: | ---: |
| ALPB TS3 seed | 1.391 | 1.245 | 1.635 | 2.082 |
| New gas result labelled TS3 | 1.229 | 1.590 | 1.879 | 1.624 |
| Existing gas TS4 | 1.236 | 1.536 | 1.847 | 1.639 |

The five reactive distance changes along the new imaginary mode are almost
identical, up to arbitrary mode sign, to those of the existing gas TS4 mode
(cosine **−0.998**). The animated motion involves the reactive core, but it
appears to be a **TS4-like saddle reached from a TS3 seed**. A single imaginary
frequency therefore does **not** establish the intended TS3 identity here.

**Review status: quarantined pending chemical review.** Please inspect whether
this geometry and mode can represent TS3 despite the TS4-like distances. If
not, it must not become the UMA gas TS3 profile reference.


## Review decision

- [ ] OptTS and NumFreq both terminated normally.
- [ ] Exactly one imaginary frequency remains.
- [ ] The animated mode moves the intended TS3 B–H/B–C reactive core, rather
      than only the pinacol or catalyst substituents.
- [ ] Reactive distances and overall geometry are chemically plausible.

Record the observed mode index, frequency, and any concern here. A failed
check means the reference remains **quarantined**; do not silently replace it
with the ALPB geometry or the ωB97 profile.